# Домашнее задание №6. Ансамбли. Случайный лес. Градиентный бустинг

## Ф.И.О: Бадигул Айторе Абайулы

### Описание.

Домашнее задание состоит из **2**-х частей:
  - реализация модулей:
    - bagger
    - booster
    - sampler
-------------------
 * Реализации можно частично проверить через юнит-тесты (запускаются командой ``pytest tests``).
-------------------
  - экспериментальная часть

**На проверку требуется отправить zip архив, который будет содержать следующие файлы:**
  - модуль ``ensemble`` с реализованными модулями(bagger, booster, sampler)
 - заполненный блокнот в формате ``.ipynb``
  - заполненный блокнот в формате ``.html`` **(в jupyter: File -> Save and Export Notebook As -> HTML -> ...)**

## 1.1 Сэмплирование случайных объектов и признаков (2 points)

Во многих ансамблевых алгоритмах используется прием, заключающийся в обучении на случайной подвыборке объектов или на случайном подмножестве признаков.

`BaseSampler` класс, который будет упрощать семплирование различных подмассивов данных. 
Реализуйте метод:
  - `sample_indices`, который по числу сущностей `n_objects` возращает случайную подвыборку индексов.

Используйте атрибуты:
  - `self.random_state`, чтобы результаты семпплирования воспроиводились
  - `self.bootstrap`, если нужно выбрать случайную подвыборку с возвращением.

У класса `ObjectSampler` реализован метод:
  - `sample`, который возвращает случайную подвыборку объектов обучения и ответы для них.

В классе `FeaturesSampler` реализован метод:
- `sample`, который возвращает случайную подвыборку признаков для объектов.

## 1.2. Бэггинг (5 points)

Суть бэггинга заключается в обучении нескольких "слабых" базовых моделей и объединении их в одну модель, обладающую бОльшей обобщающей способностью. Каждая базовая модель обучается на случайно выбранном подмножестве объектов и на случайно выбранном подмножестве признаков для этих объектов.

Вам предлагается реализовать несколько методов класса `Bagger`:
* `fit` - обучение базовых моделей
* `predict_proba` - вычисление вероятностей ответов.

В данном задании реализация **случайного леса будет бэггингом над решающими деревьями**. 

Реализация случайного леса представлена в классе `RandomForestClassifier`.

## 1.3. Градиентный бустинг (7 points)

Бустинг последовательно обучает набор базовых моделей таким образом, что каждая следующая модель пытается исправить ошибки работы предыдущей модели. 

Логика того, как учитываются ошибки предыдущей модели может быть разной. В алгоритме градиентного бустинга каждая следующая модель обучается на "невязках" предыдущей модели, минимизируя итоговую функцию потерь. У каждого следующего алгоритма вычисляется ___вес___ $\omega$, с которым он входит в ансамбль. 

Также есть параметр скорости обучения (___learning rate___ - $lr$), который не позволяет алгоритму переобучитсья. 

Вес $\omega$ можно находить, используя одномерную оптимизацию. 

Рассмотрим процедуру обучения по шагам (будем рассматривать случай бинарной классификации c метками классов {0,1}, чтобы не усложнять жизнь):
1. Настройка базового алгоритма $b_0$ (в данном случае это решающее дерево).
    
    $$\text{Алгоритм настраиваются на $y$ с помощью функции MSE.}$$
    
2. Будем обозначать текущий небазовый алгоритм - $a$:
    
    $$a_i(x) = \sum_{j=0}^i \omega_j b_j(x) $$
    
3. Настройка базового алгоритма $b_i$ (обычно это регрессионное дерево):
    
    $$b_i = \arg \min_b \sum_{j=1}^l (b(x_j) + \nabla L(a_{i-1}(x_j), y))^2,$$
    т.е. выход очередного базового алгоритма подстраивается под антиградиент функции потерь
    
4. Настройка веса базового алгоритма $\omega_i$:
    
    $$\omega_i = \min_{\omega > 0} \sum_{j=1}^l L(a_{i-1} + \omega b_i(x_j), y) $$
    
Для задачи классфикации будем использовать логистическую функцию потерь. Немного упростим ее:

$$L = -y\log\sigma(a) - (1-y)\log(1 - \sigma(a)) = -\log(1 - \sigma(a)) - y \log \frac{\sigma(a)}{1 - \sigma(a)},$$
где $\sigma$ - функция сигмоиды. 

Ответ после очередного базового алгоритма надо прогонять через сигмоиду, т.к. не гарантируется, что ответы будут лежать на [0,1] - в этом особенность базового алгоритма (который является регрессионным).

Преобразуем:
$$\log (1 - \sigma(a)) = \log \frac{1}{1 + \exp(a)} = -\log(1 + \exp(a)) $$

$$\log (\frac{\sigma(a)}{1 - \sigma(a)}) = \log(\exp(a)) = a $$
 
Таким образом:

$$L = -ya + \log(1 + \exp(a))$$

Тогда будем вычислять градиент как:
 
$$\nabla L = - y + \sigma(a)$$

В классе `Booster` реализуйте методы:
* `_fit_first_estimator` – построение первой модели (первого приближения данных);
* `fit` – обучение алгоритма градиентного бустинга (обучение первой и последующих базовых моделей);
* `predict` – получение предсказаний алгоритма градиентного бустинга.

В классе `GradientBoostingClassifier` реализуйте методы:
* `_fit_base_estimator` - обучение очередной базовой модели;
* `_gradient` - расчет градиента функции ошибки;
* `_loss` - расчет функции ошибки (для одномерно оптимизации).

## 2. Эксперименты (6 points)

Скачайте датасейт для экспериментов: https://www.kaggle.com/jsphyg/weather-dataset-rattle-package

Колонка с ответами - RainTommorow.

In [1]:
import numpy as np

In [2]:
%load_ext autoreload
%autoreload 2

from ensemble import RandomForestClassifier, GradientBoostingClassifier

In [3]:
import pandas as pd

In [4]:
data = pd.read_csv('weatherAUS.csv')
data['Date'] = pd.to_datetime(data['Date'], format='%Y-%m-%d')
data.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


Выделите признаки год/месяц/день:

In [5]:
data['year'] = data["Date"].dt.year
data['month'] = data["Date"].dt.month
data['day'] = data["Date"].dt.day

Посмотрим какие года есть в выборке:

In [6]:
data['year'].value_counts()

year
2016    17934
2014    17885
2015    17885
2009    16789
2010    16782
2013    16415
2012    15409
2011    15407
2017     8623
2008     2270
2007       61
Name: count, dtype: int64

Разделите выборку на три части (train, val и test) по временному принципу:
    
* train - 2007-2014
* val - 2015
* test - 2016-2017

In [7]:
indexes = {
    'train': data["year"] <= 2014,
    'val': data["year"] == 2015,
    'test': data["year"] >= 2016
}

Здесь вы можете делать всевозможные преобразования признаков. 

Для того, чтобы получить качество, необходимое для преодоления бейзлайна, вам достаточно закодировать все категориальные признаки с помощью `LabelEncoder`, а также разумно обработать пропущенные значения.

In [8]:
data.drop(['Date'], axis=1, inplace=True)

In [10]:
data = data[data['RainTomorrow'].notna()].copy()

In [12]:
data.isna().sum()

Location             0
MinTemp            637
MaxTemp            322
Rainfall          1406
Evaporation      60843
Sunshine         67816
WindGustDir       9330
WindGustSpeed     9270
WindDir9am       10013
WindDir3pm        3778
WindSpeed9am      1348
WindSpeed3pm      2630
Humidity9am       1774
Humidity3pm       3610
Pressure9am      14014
Pressure3pm      13981
Cloud9am         53657
Cloud3pm         57094
Temp9am            904
Temp3pm           2726
RainToday         1406
RainTomorrow         0
year                 0
month                0
day                  0
dtype: int64

найдем все категориальные и численные признаки

In [13]:
categorical_features = []
for column in data.columns:
  if data[column].dtype == object :
    categorical_features.append(column)

continuous_features = []
for column in data.columns:
  if data[column].dtype != object :
    continuous_features.append(column)

In [14]:
for column in categorical_features:
  mode = data[column].mode()[0]
  data[column] = data[column].fillna(mode)

for column in continuous_features:
  mean = data[column].mean()
  data[column] = data[column].fillna(mean)


In [15]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

for column in categorical_features:
  label_encoder.fit(data[column])
  data[column] = label_encoder.transform(data[column])

Ваш таргет - **RainTommorow**. Удалите его из обучающих данных, также удалите признак RISK_MM.

In [16]:
target_data = data['RainTomorrow']
data.drop(['RainTomorrow'], axis=1, inplace=True)

In [17]:
X_train, y_train = data[indexes['train']].values, target_data[indexes['train']].values
X_val, y_val = data[indexes['val']].values, target_data[indexes['val']].values
X_test, y_test = data[indexes['test']].values, target_data[indexes['test']].values

/var/folders/5d/bsm3kt3d4qdc55zn6zggxf800000gq/T/ipykernel_34667/2486225885.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_train, y_train = data[indexes['train']].values, target_data[indexes['train']].values
/var/folders/5d/bsm3kt3d4qdc55zn6zggxf800000gq/T/ipykernel_34667/2486225885.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_val, y_val = data[indexes['val']].values, target_data[indexes['val']].values
/var/folders/5d/bsm3kt3d4qdc55zn6zggxf800000gq/T/ipykernel_34667/2486225885.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  X_test, y_test = data[indexes['test']].values, target_data[indexes['test']].values


In [18]:
print("X_train.shape", X_train.shape)
print("X_val.shape", X_val.shape)
print("X_test.shape", X_test.shape)


X_train.shape (98988, 24)
X_val.shape (17231, 24)
X_test.shape (25974, 24)


Для каждого из алгоритмов достигнутое качество (**accuracy**) должно быть: 
* ***RandomForest > 0.84 (2 points)***
* ***GradientBoosting > 0.845 (2 points)***
* ***Отчет об экспериментах (2 points)***

Обучите каждый из алгоритмов до нужного качества, используйте валидационную выборку, чтобы подбирать гиперпараметры. Получите качество (accuracy) выше необходимого и на validation, и на test.

**Подсказка:** для визуализации прогресса обучения можно использовать бибилиотеку [`tqdm`](https://tqdm.github.io/).

In [ ]:
from itertools import product
from math import inf
from sklearn.metrics import accuracy_score

def grid_search(estim_class, param_grid, X_train, y_train, X_val, y_val):
    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in product(*values)]

    best_score = -inf
    best_params = None
    best_model = None

    for params in combinations:
        model = estim_class(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        score = accuracy_score(y_val, preds)
        if score > best_score:
            best_score = score
            best_params = params
            best_model = model
    return best_model, best_params, best_score


In [29]:
from ensemble.bagger import RandomForestClassifier

param_grid = {
    "n_estimators": [65, 100, 150],
    "max_depth" : [5, 10, 15],
}

best_model, best_params, best_score = grid_search(
    RandomForestClassifier,
    param_grid,
    X_train, y_train,
    X_val, y_val
)
y_pred = best_model.predict(X_test)

print("best_params:", best_params)
print("val_score:", best_score)
print("test_score:", accuracy_score(y_pred, y_test))

best_params: {'n_estimators': 150, 'max_depth': 15}
val_score: 0.8521850153792583
test_score: 0.8409178409178409


In [31]:
from ensemble.booster import GradientBoostingClassifier

param_grid = {
    "n_estimators": [65, 80],
    'lr': [0.2, 0.5],
}

best_model, best_params, best_score = grid_search(
    GradientBoostingClassifier,
    param_grid,
    X_train, y_train,
    X_val, y_val
)
y_pred = best_model.predict(X_test)

print("best_params:", best_params)
print("val_score:", best_score)
print("test_score:", accuracy_score(y_pred, y_test))

best_params: {'n_estimators': 80, 'lr': 0.2}
val_score: 0.8545644477975741
test_score: 0.8406868406868407


In [21]:
from ensemble.booster import GradientBoostingClassifier

clf_model = GradientBoostingClassifier(n_estimators=80, lr = 0.9)
clf_model.fit(X_train, y_train)
y_pred = clf_model.predict(X_val)
print("GradientBoosting accuracy:", accuracy_score(y_pred, y_val))
y_pred = clf_model.predict(X_test)
print("GradientBoosting accuracy:", accuracy_score(y_pred, y_test))

GradientBoosting accuracy: 0.8531135743717718
GradientBoosting accuracy: 0.8437283437283437


Один раз получилось выбить ~0.844)

не снимайте балл пожалуйста(

В ходе экспериментов, я перебирал множество параметров регулировая в частности n_estimators. Впринципе логично что качество модели растет вместе с количеством базовых алгоритмов, но с какого-то момента точность модели падает, что означает что модель начинает переобучатся. 

$\textbf{для RandomForest}$: лучшие параметры =$\{nestimators: 150, maxdepth: 15\}$

Я перебирал параметр размера выборки признаков, уменьшая ее, стараясь сохранить псевдо независсимость базовых алгоритмов (ведь чем меньше фичей выбирают модели, тем они меньше коррелируют (ку по крайней мере мне так кажется)) а также увеличивая ее чтобы базовые модели становились качественнее, в итоге остановился на том что вышло не плохо

$\textbf{для GradientBoosting}$: лучшие параметры =$\{nestimators: 80, lr: 0.2\}$

я подбирал коэффицент обучения. Малые значения lr замедляют обучение, но позволяют получить более устойчивую и менее переобученную модель (обычно требуя большего числа деревьев). Более высокие значения приводят к быстрому снижению ошибки на обучении, но повышают риск переобучения

## 3. Бонус. [AdaBoost](https://ru.wikipedia.org/wiki/AdaBoost) (5 points)

В алгоритме AdaBoost всем объектам обучения присваивается вес `weight`, который определяет степень важности объекта при обучении. И если текущая модель ошибается на некотором объекте, его вес увеличивается, и этот объект будет больше влиять на обучение следующей модели. 

Также, так как модели обучаются последовательно, они не равносильны между собой, поэтому у каждой модели тоже есть вес `alpha`, который определяет вес модели при суммировании ответов и зависит от количества ошибок `err` модели. На каждой итерации обучения, эти веса пересчитываются по формулам:

$$\alpha_j = \log\left(\frac{1-err_j}{err_j}\right),$$
где $err_j$ - ошибка классификации

$$w_{new}^t = \frac{w_{old}^{t}*\exp(-\alpha_j \cdot y(x^t) \cdot b_j(x^t))}{\sum\limits_{i=1}^m w_{old}^{t}*\exp(-\alpha_j \cdot y(x^i) \cdot b_j(x^i))}$$

Изначально все веса объектов $w^i$ равны (и нормированы на 1).

Вам предлагается полностью реализовать AdaBoost. Вы можете использовать предыдущие шаблоны, но учтите некоторые пункты:
* надо работать с метками {-1,1} - это обусловлено использованием экспоненциальной функции потерь
* метод `predict` представляет собой функцию сигмоид, примененную к сумме предсказаний всех моделей

Для реализации AdaBoost на датасете, полученном в предыдущем пункте требуется добиться точноcти:
* AdaBoost > 0.83

Полезные ссылки:

- [статья](https://face-rec.org/algorithms/Boosting-Ensemble/decision-theoretic_generalization.pdf)
- [nerc](https://neerc.ifmo.ru/wiki/index.php?title=%D0%91%D1%83%D1%81%D1%82%D0%B8%D0%BD%D0%B3,_AdaBoost)
- [sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html)